# Games Backtest (2025 default)

Backtest variant of `games_today_tomorrow.ipynb`:

- Defaults to **completed 2025 games** from `data/games_2025.parquet`.
- Keeps V6-style scoring logic, but changes data selection upstream.
- If Kalshi historical implied columns are missing, scoring still runs and Kalshi-based metrics are omitted (set to NaN).

Run top-to-bottom after refreshing Parquet files.

In [3]:
# Backtest parameters (edit these)
SEASON = 2025
START_DATE = "2025-03-27"  # inclusive, YYYY-MM-DD
END_DATE = "2025-11-11"             # inclusive; None = today

FINAL_STATES = {"Final", "Game Over", "Completed Early"}

print({
    "SEASON": SEASON,
    "START_DATE": START_DATE,
    "END_DATE": END_DATE or "today",
    "FINAL_STATES": sorted(FINAL_STATES),
})

{'SEASON': 2025, 'START_DATE': '2025-03-27', 'END_DATE': '2025-11-11', 'FINAL_STATES': ['Completed Early', 'Final', 'Game Over']}


In [4]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

cwd = Path.cwd()
DATA = cwd / "data" if (cwd / "data").is_dir() else cwd.parent / "data"
PARQUET = DATA / f"games_{SEASON}.parquet"

if not PARQUET.is_file():
    raise FileNotFoundError(f"Missing {PARQUET}. Run: python -m app.main live --season {SEASON}")

df = pd.read_parquet(PARQUET)
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce").dt.normalize()
df["detailed_state"] = df["detailed_state"].astype(str)

df_bt = df[df["detailed_state"].isin(FINAL_STATES)].copy()

if "home_win" in df_bt.columns:
    # ensure final training label is present for backtest
    df_bt = df_bt[df_bt["home_win"].notna()].copy()

start_ts = pd.Timestamp(START_DATE).normalize()
end_ts = pd.Timestamp.today().normalize() if END_DATE is None else pd.Timestamp(END_DATE).normalize()
if end_ts < start_ts:
    raise ValueError(f"END_DATE ({end_ts.date()}) must be >= START_DATE ({start_ts.date()})")

sub = df_bt[df_bt["game_date"].between(start_ts, end_ts)].copy()

print("Using:", PARQUET.resolve())
print(f"Rows total={len(df)}  completed={len(df_bt)}")
if len(df_bt):
    print(f"Date range completed: {df_bt['game_date'].min().date()} .. {df_bt['game_date'].max().date()}")
print(f"Backtest window: {start_ts.date()} .. {end_ts.date()}  rows={len(sub)}")


Using: C:\Users\Austin\baseball\mlb-model\data\games_2025.parquet
Rows total=3013  completed=2894
Date range completed: 2025-03-01 .. 2025-11-01
Backtest window: 2025-03-27 .. 2025-11-11  rows=2581


In [5]:
# Core thresholds from your mismatch workflow
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
MIN_CORE_MATCHES = 2

USE_PLATOON_EXTRA = True
PLATOON_MIN = 0.0
USE_PEN_EXTRA = True
PEN_ROLL_MIN_GAP = 0.0

s = sub.copy()
req = {"sp_kbb_diff", "sp_xfip_diff", "offense_diff"}
missing = req - set(s.columns)
if missing:
    raise ValueError(f"Missing columns for mismatch filter: {missing}")

# Core directional checks
home_core_kbb = s["sp_kbb_diff"] > SP_KBB_ABS
home_core_xfip = s["sp_xfip_diff"] < -SP_XFIP_ABS
home_core_off = s["offense_diff"] > OFFENSE_ABS
home_core_n = home_core_kbb.astype(int) + home_core_xfip.astype(int) + home_core_off.astype(int)

away_core_kbb = s["sp_kbb_diff"] < -SP_KBB_ABS
away_core_xfip = s["sp_xfip_diff"] > SP_XFIP_ABS
away_core_off = s["offense_diff"] < -OFFENSE_ABS
away_core_n = away_core_kbb.astype(int) + away_core_xfip.astype(int) + away_core_off.astype(int)

home_base = home_core_n >= MIN_CORE_MATCHES
away_base = away_core_n >= MIN_CORE_MATCHES

home_ext = home_base.copy()
away_ext = away_base.copy()

if USE_PLATOON_EXTRA and "offense_platoon_diff" in s.columns:
    po = s["offense_platoon_diff"].fillna(0)
    home_ext = home_ext & (po > PLATOON_MIN)
    away_ext = away_ext & (po < -PLATOON_MIN)

if USE_PEN_EXTRA and {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(s.columns):
    hr = s["home_pen_roll14_fip"]
    ar = s["away_pen_roll14_fip"]
    g = PEN_ROLL_MIN_GAP
    pen_ok_home = (hr + g < ar) | hr.isna() | ar.isna()
    pen_ok_away = (ar + g < hr) | hr.isna() | ar.isna()
    home_ext = home_ext & pen_ok_home
    away_ext = away_ext & pen_ok_away

favorites = s.copy().sort_values(["game_date", "home_team_name"]).reset_index(drop=True)
favorites["home_core_matches"] = home_core_n.astype(int)
favorites["away_core_matches"] = away_core_n.astype(int)
favorites["home_base_match"] = home_base
favorites["away_base_match"] = away_base
favorites["home_with_extras"] = home_ext
favorites["away_with_extras"] = away_ext

_tie_home = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + (-favorites["sp_xfip_diff"]).abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
_tie_away = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
prefer_home = (favorites["home_core_matches"] > favorites["away_core_matches"]) | (
    (favorites["home_core_matches"] == favorites["away_core_matches"]) & (_tie_home >= _tie_away)
)

favorites["_mismatch_side"] = prefer_home.map({True: "home_favored", False: "away_favored"})
favorites["core_matches"] = favorites[["home_core_matches", "away_core_matches"]].max(axis=1)
favorites["favored_team"] = np.where(
    favorites["_mismatch_side"].eq("home_favored"),
    favorites["home_team_name"],
    favorites["away_team_name"],
)

print("Favorites frame rows:", len(favorites))
display(favorites.tail(25))

Favorites frame rows: 2581


,game_pk,game_date,detailed_state,home_team_name,away_team_name,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher,away_probable_pitcher,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_kbb_diff,sp_xfip_diff,offense_diff,home_offense_platoon,away_offense_platoon,offense_platoon_diff,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_fip,away_pen_season_fip,home_pen_season_era,away_pen_season_era,home_pen_roll14_fip,away_pen_roll14_fip,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,_mismatch_side,core_matches,favored_team
2556,813055,2025-10-08,Final,Detroit Tigers,Seattle Mariners,116,136,9.0,3.0,Casey Mize,Bryce Miller,663554,682243,R,R,101.461378,102.992345,16.427432,10.204082,3.851678,5.136900,1.0,6.223351,-1.285223,-1.530967,98.233996,103.062914,-4.828918,33.333333,0.000000,0.573684,10.100000,4.075678,3.777122,3.583739,3.636531,1.651724,2.860000,1.396552,4.320000,-2.423953,-0.917122,NaN,NaN,NaN,2025,0.0,0.0,False,False,False,False,home_favored,0.0,Detroit Tigers
2557,813041,2025-10-08,Final,Los Angeles Dodgers,Philadelphia Phillies,119,143,2.0,8.0,Yoshinobu Yamamoto,Aaron Nola,808967,605400,R,R,106.889353,105.636743,20.760234,17.079208,2.904223,4.541696,0.0,3.681026,-1.637473,1.252610,106.236203,105.408389,0.827815,20.833333,34.615385,1.766667,2.475000,3.881964,4.100000,4.002471,4.142857,1.864706,2.561538,2.117647,2.076923,-2.017258,-1.538462,NaN,NaN,NaN,2025,2.0,0.0,True,False,False,False,home_favored,2.0,Los Angeles Dodgers
2558,813060,2025-10-08,Final,New York Yankees,Toronto Blue Jays,147,141,2.0,5.0,Cam Schlittler,Louis Varland,693645,686973,R,R,109.533751,105.775922,17.434211,17.905405,3.702740,3.100000,0.0,-0.471195,0.602740,3.757829,108.029801,105.270419,2.759382,30.769231,36.363636,1.814286,0.766667,3.758587,4.008023,3.982687,3.980431,2.933333,3.100000,0.000000,1.000000,-0.825254,-0.908023,NaN,NaN,NaN,2025,0.0,2.0,False,True,False,True,away_favored,2.0,Toronto Blue Jays
2559,813050,2025-10-09,Final,Chicago Cubs,Milwaukee Brewers,112,158,6.0,0.0,Matthew Boyd,Freddy Peralta,571510,642547,L,R,104.384134,102.296451,15.598886,19.087137,3.612059,3.609434,1.0,-3.488251,0.002625,2.087683,103.614790,104.365136,-0.750346,NaN,33.333333,NaN,6.600000,3.967769,3.396875,3.849174,2.973214,1.728571,2.700000,2.314286,0.900000,-2.239197,-0.696875,NaN,NaN,NaN,2025,1.0,0.0,False,False,False,False,home_favored,1.0,Chicago Cubs
2560,813042,2025-10-09,Final,Los Angeles Dodgers,Philadelphia Phillies,119,143,2.0,1.0,Tyler Glasnow,Cristopher Sánchez,607192,650911,R,L,106.889353,105.636743,17.213115,20.817844,3.719926,2.515842,1.0,-3.604729,1.204085,1.252610,108.772928,105.408389,3.364539,15.384615,44.444444,2.100000,0.276471,3.881964,4.100000,4.002471,4.142857,1.242857,2.463636,0.642857,2.454545,-2.639107,-1.636364,NaN,NaN,NaN,2025,0.0,1.0,False,False,False,False,away_favored,1.0,Philadelphia Phillies
2561,813054,2025-10-10,Final,Seattle Mariners,Detroit Tigers,136,116,3.0,2.0,George Kirby,Tarik Skubal,669923,669373,R,L,102.992345,101.461378,20.610687,27.807487,3.330159,2.413993,1.0,-7.196800,0.916166,1.530967,103.369828,98.233996,5.135832,45.000000,NaN,2.900000,NaN,3.777122,4.075678,3.636531,3.583739,2.900000,1.745161,4.500000,1.741935,-0.877122,-2.330516,NaN,NaN,NaN,2025,0.0,1.0,False,False,False,False,away_favored,1.0,Detroit Tigers
2562,813046,2025-10-11,Final,Milwaukee Brewers,Chicago Cubs,158,112,3.0,1.0,Trevor Megill,Drew Pomeranz,656730,519141,R,L,102.296451,104.384134,22.395833,20.689655,2.461702,3.321477,1.0,1.706178,-0.859774,-2.087683,104.365136,103.614790,0.750346,66.666667,66.666667,-0.900000,-0.900000,3.396875,3.967769,2.9

In [6]:
# V6 model block (kept aligned in spirit with games_today_tomorrow)
s = favorites.copy()

# Stable scaling
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
PLATOON_ABS = 10.0
PEN_ABS = 0.75

kbb_norm = s["sp_kbb_diff"] / SP_KBB_ABS
xfip_norm = -s["sp_xfip_diff"] / SP_XFIP_ABS
off_norm = s["offense_diff"] / OFFENSE_ABS
platoon_norm = s["offense_platoon_diff"].fillna(0) / PLATOON_ABS

# Pen: prefer roll14; neutralize missing values so matrix math is stable
if "home_pen_roll14_fip" in s.columns:
    hr_pen = s["home_pen_roll14_fip"]
    if "home_pen_season_fip" in s.columns:
        hr_pen = hr_pen.combine_first(s["home_pen_season_fip"])
else:
    hr_pen = pd.Series(np.nan, index=s.index)

if "away_pen_roll14_fip" in s.columns:
    ar_pen = s["away_pen_roll14_fip"]
    if "away_pen_season_fip" in s.columns:
        ar_pen = ar_pen.combine_first(s["away_pen_season_fip"])
else:
    ar_pen = pd.Series(np.nan, index=s.index)

pen_gap = hr_pen - ar_pen  # home - away; lower home FIP better
pen_norm = (-pen_gap / PEN_ABS).fillna(0.0)

sp_vector = (0.65 * kbb_norm) + (0.35 * xfip_norm)
off_vector = (0.70 * off_norm) + (0.30 * platoon_norm)
pen_vector = pen_norm

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])
sign_matrix = np.sign(signal_matrix)
mag_matrix = np.abs(signal_matrix)

mean_sign = np.mean(sign_matrix, axis=0)
mean_mag = np.mean(mag_matrix, axis=0)

agreement = 1 - np.mean(np.var(sign_matrix, axis=0))
direction_purity = np.abs(mean_sign)
mag_consistency = 1 / (1 + np.std(signal_matrix, axis=0))

coherence_score = (0.45 * agreement) + (0.30 * direction_purity) + (0.25 * mag_consistency)
raw_edge = mean_mag
instability = np.std(signal_matrix, axis=0)

direction_conflict = (
    (np.sign(sp_vector) != np.sign(off_vector)).astype(int)
    + (np.sign(sp_vector) != np.sign(pen_vector)).astype(int)
    + (np.sign(off_vector) != np.sign(pen_vector)).astype(int)
)

risk_penalty = (
    0.35 * direction_conflict
    + 0.45 * instability
    + 0.20 * np.abs(sp_vector - off_vector)
)
risk_penalty = np.tanh(risk_penalty)

trap_score = raw_edge * (1 - coherence_score)
quality_score = raw_edge * coherence_score * np.exp(-1.25 * instability) * (1 / (1 + risk_penalty))
risk_adjusted_edge = quality_score - 0.5 * trap_score

prefer_home = sp_vector >= 0
s["favored_team"] = np.where(prefer_home, s["home_team_name"], s["away_team_name"])
s["_mismatch_side"] = np.where(prefer_home, "home_favored", "away_favored")

s["signal_agreement"] = np.sum(sign_matrix > 0, axis=0)
s["signal_conflict"] = np.sum(sign_matrix < 0, axis=0)
s["direction_conflict"] = direction_conflict
s["instability"] = instability
s["coherence_score"] = coherence_score
s["quality_score"] = quality_score
s["risk_adjusted_edge"] = risk_adjusted_edge
s["trap_score"] = trap_score
s["risk_penalty"] = risk_penalty

s["core_matches"] = (
    (np.abs(kbb_norm) > 1).astype(int)
    + (np.abs(xfip_norm) > 1).astype(int)
    + (np.abs(off_norm) > 1).astype(int)
)

# Backtest outcome: did the model's favored side win?
if "home_win" in s.columns:
    s["favorite_won"] = np.where(
        prefer_home,
        s["home_win"] == 1.0,
        s["home_win"] == 0.0,
    )
else:
    s["favorite_won"] = np.nan

# Kalshi is optional in historical backtests.
if {"kalshi_home_implied", "kalshi_away_implied"}.issubset(s.columns):
    s["implied_prob"] = np.where(prefer_home, s["kalshi_home_implied"], s["kalshi_away_implied"])
    s["market_edge"] = s["risk_adjusted_edge"] - s["implied_prob"]
else:
    s["implied_prob"] = np.nan
    s["market_edge"] = np.nan

scored = s.sort_values(["risk_adjusted_edge", "coherence_score"], ascending=[False, False]).reset_index(drop=True)

show_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "_mismatch_side", "favorite_won",
    "risk_adjusted_edge", "quality_score", "coherence_score", "instability",
    "signal_agreement", "signal_conflict", "core_matches",
    "sp_kbb_diff", "sp_xfip_diff", "offense_diff", "offense_platoon_diff",
    "implied_prob", "market_edge", "home_win",
]
show_cols = [c for c in show_cols if c in scored.columns]

print(f"Scored games: {len(scored)}")
display(scored[show_cols].head(25))

# Optional aggregate view (uncomment to use):
# display(
#     scored["favorite_won"]
#     .map({True: "Favorite Won", False: "Favorite Did Not Win"})
#     .fillna("Unknown")
#     .value_counts(dropna=False)
#     .rename_axis("favorite_result")
#     .to_frame("count")
# )

Scored games: 2581


,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,favorite_won,risk_adjusted_edge,quality_score,coherence_score,instability,signal_agreement,signal_conflict,core_matches,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,implied_prob,market_edge,home_win
0,2025-04-06,Pittsburgh Pirates,New York Yankees,New York Yankees,away_favored,False,0.625929,0.968371,0.683944,0.203132,0,3,3,-6.326490,1.522131,-18.371608,-21.987365,NaN,NaN,1.0
1,2025-08-28,Chicago White Sox,New York Yankees,New York Yankees,away_favored,True,0.594559,0.821989,0.696088,0.136699,0,3,3,-5.590776,0.569815,-15.588031,-16.142384,NaN,NaN,0.0
2,2025-05-18,Chicago Cubs,Chicago White Sox,Chicago Cubs,home_favored,True,0.570910,0.737326,0.713128,0.054964,3,0,2,2.237335,-1.017705,10.438413,11.727373,NaN,NaN,1.0
3,2025-09-02,St. Louis Cardinals,Athletics,Athletics,away_favored,False,0.430632,0.548220,0.716891,0.038470,0,3,1,-1.079414,0.928475,-7.794015,-8.278146,NaN,NaN,1.0
4,2025-06-29,Texas Rangers,Seattle Mariners,Seattle Mariners,away_favored,True,0.405208,0.517317,0.715859,0.042943,0,3,1,-3.031770,0.273687,-7.933194,-7.174393,NaN,NaN,0.0
5,2025-05-02,New York Yankees,Tampa Bay Rays,New York Yankees,home_favored,True,0.401588,0.567463,0.693512,0.150169,3,0,2,1.617271,-1.289217,10.160056,13.191183,NaN,NaN,1.0
6,2025-06-02,Boston Red Sox,Los Angeles Angels,Boston Red Sox,home_favored,False,0.400573,0.511094,0.715390,0.044989,3,0,1,4.461282,0.202119,6.958942,10.032235,NaN,NaN,0.0
7,2025-05-14,New York Mets,Pittsburgh Pirates,New York Mets,home_favored,False,0.397490,0.595216,0.684137,0.202016,3,0,2,2.141548,-0.831361,13.639527,7.626495,NaN,NaN,0.0
8,2025-06-28,New York Yankees,Athletics,New York Yankees,home_favored,False,0.350445,0.459578,0.709825,0.069873,3,0,1,-0.286059,-1.265065,5.288796,9.570147,NaN,NaN,0.0
9,2025-08-10,Arizona Diamondbacks,Colorado Rockies,Arizona Diamondbacks,home_favored,True,0.339929,0.481323,0.695536,0.139558,3,0,2,0.988475,-0.830372,10.995129,11.727373,NaN,NaN,1.0


In [7]:
# Additional breakdowns by metric bands
band_df = scored.copy()

# Keep known outcomes for cleaner win-rate summaries
band_df = band_df[band_df["favorite_won"].isin([True, False])].copy()

band_specs = {
    "risk_adjusted_edge": [-np.inf, 0.25, 0.50, 0.75, 1.00, np.inf],
    "quality_score": [-np.inf, 0.10, 0.20, 0.30, 0.40, np.inf],
    "coherence_score": [-np.inf, 0.40, 0.50, 0.60, 0.70, np.inf],
    "instability": [-np.inf, 0.80, 1.00, 1.20, 1.50, np.inf],
}

for metric, bins in band_specs.items():
    if metric not in band_df.columns:
        continue

    d = band_df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        continue

    d["band"] = pd.cut(d[metric], bins=bins, include_lowest=True)

    out = (
        d.groupby("band", observed=False)["favorite_won"]
        .agg(
            games="count",
            favorite_wins=lambda x: int((x == True).sum()),
            favorite_losses=lambda x: int((x == False).sum()),
            favorite_win_rate="mean",
        )
        .reset_index()
    )
    out["favorite_win_rate"] = out["favorite_win_rate"].round(3)

    print(f"\n{metric} band breakdown")
    display(out)



risk_adjusted_edge band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.25]",2553,1441,1112,0.564
1,"(0.25, 0.5]",25,11,14,0.440
2,"(0.5, 0.75]",3,2,1,0.667
3,"(0.75, 1.0]",0,0,0,NaN
4,"(1.0, inf]",0,0,0,NaN



quality_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.1]",1950,1068,882,0.548
1,"(0.1, 0.2]",372,223,149,0.599
2,"(0.2, 0.3]",163,99,64,0.607
3,"(0.3, 0.4]",59,42,17,0.712
4,"(0.4, inf]",37,22,15,0.595



coherence_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.4]",1163,641,522,0.551
1,"(0.4, 0.5]",605,313,292,0.517
2,"(0.5, 0.6]",336,220,116,0.655
3,"(0.6, 0.7]",468,276,192,0.590
4,"(0.7, inf]",9,4,5,0.444



instability band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.8]",753,410,343,0.544
1,"(0.8, 1.0]",304,171,133,0.562
2,"(1.0, 1.2]",333,191,142,0.574
3,"(1.2, 1.5]",369,210,159,0.569
4,"(1.5, inf]",822,472,350,0.574


In [8]:
# Decision / confidence layers + backtest summary
def decision_layer(df: pd.DataFrame) -> pd.Series:
    conditions = [
        (df["risk_adjusted_edge"] > 1.0) & (df["coherence_score"] > 0.60) & (df["instability"] < 1.2),
        (df["risk_adjusted_edge"] > 0.55) & (df["coherence_score"] > 0.45),
    ]
    choices = ["BET", "LEAN"]
    return np.select(conditions, choices, default="PASS")


def confidence_tier(df: pd.DataFrame) -> pd.Series:
    return np.where(
        (df["decision"] == "BET") & (df["coherence_score"] > 0.75),
        "A (Strong Bet)",
        np.where(
            df["decision"] == "BET",
            "B (Playable Bet)",
            np.where(df["decision"] == "LEAN", "C (Lean Only)", "D (Pass)"),
        ),
    )


bt = scored.copy()
bt["decision"] = decision_layer(bt)
bt["tier"] = confidence_tier(bt)

# Evaluate pick correctness only on non-PASS rows
bt_eval = bt[bt["decision"].isin(["BET", "LEAN"])].copy()
bt_eval["pick_home"] = bt_eval["_mismatch_side"].eq("home_favored")
bt_eval["won"] = np.where(bt_eval["pick_home"], bt_eval["home_win"] == 1.0, bt_eval["home_win"] == 0.0)

summary = {
    "rows_scored": int(len(bt)),
    "rows_bet_or_lean": int(len(bt_eval)),
    "win_rate_bet_or_lean": float(bt_eval["won"].mean()) if len(bt_eval) else np.nan,
    "bets_only": int((bt["decision"] == "BET").sum()),
    "leans_only": int((bt["decision"] == "LEAN").sum()),
}

print(summary)
if len(bt_eval):
    by_tier = bt_eval.groupby("tier")["won"].agg(["count", "mean"]).rename(columns={"mean": "win_rate"})
    display(by_tier.sort_values("count", ascending=False))

view_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "decision", "tier",
    "risk_adjusted_edge", "coherence_score", "instability", "home_win", "won",
]
view_cols = [c for c in view_cols if c in bt_eval.columns]
display(bt_eval.sort_values("risk_adjusted_edge", ascending=False)[view_cols].head(50))

{'rows_scored': 2581, 'rows_bet_or_lean': 3, 'win_rate_bet_or_lean': 0.6666666666666666, 'bets_only': 0, 'leans_only': 3}


,count,win_rate
tier,,
C (Lean Only),3,0.666667


,game_date,home_team_name,away_team_name,favored_team,decision,tier,risk_adjusted_edge,coherence_score,instability,home_win,won
0,2025-04-06,Pittsburgh Pirates,New York Yankees,New York Yankees,LEAN,C (Lean Only),0.625929,0.683944,0.203132,1.0,False
1,2025-08-28,Chicago White Sox,New York Yankees,New York Yankees,LEAN,C (Lean Only),0.594559,0.696088,0.136699,0.0,True
2,2025-05-18,Chicago Cubs,Chicago White Sox,Chicago Cubs,LEAN,C (Lean Only),0.570910,0.713128,0.054964,1.0,True


In [9]:
# Quantile calibration tables + monotonicity checks

cal_df = scored.copy()
if "favorite_won" not in cal_df.columns:
    raise ValueError("favorite_won missing; run V6 scoring cell first.")

cal_df = cal_df[cal_df["favorite_won"].isin([True, False])].copy()
if cal_df.empty:
    raise ValueError("No rows with known favorite_won values for calibration.")

metrics = [
    "risk_adjusted_edge",
    "quality_score",
    "coherence_score",
    "instability",
]


def quantile_calibration_table(df: pd.DataFrame, metric: str, q: int = 10, ascending_good: bool = True):
    d = df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        return None, None

    # Handle low-variance metrics safely.
    n_unique = d[metric].nunique(dropna=True)
    q_eff = max(2, min(q, int(n_unique)))

    d["quantile"] = pd.qcut(d[metric], q=q_eff, duplicates="drop")
    tab = (
        d.groupby("quantile", observed=False)["favorite_won"]
        .agg(games="count", favorite_wins="sum", favorite_win_rate="mean")
        .reset_index()
    )
    tab["favorite_losses"] = tab["games"] - tab["favorite_wins"]
    tab["favorite_win_rate"] = tab["favorite_win_rate"].round(3)

    rates = tab["favorite_win_rate"].to_numpy()
    if not ascending_good:
        rates = rates[::-1]
    deltas = np.diff(rates)
    mono_score = float((deltas >= 0).mean()) if len(deltas) else np.nan
    is_monotonic = bool((deltas >= 0).all()) if len(deltas) else True

    summary = {
        "metric": metric,
        "quantile_bins": int(len(tab)),
        "rows_used": int(tab["games"].sum()),
        "win_rate_first_bin": float(tab.iloc[0]["favorite_win_rate"]),
        "win_rate_last_bin": float(tab.iloc[-1]["favorite_win_rate"]),
        "lift_last_minus_first": float(tab.iloc[-1]["favorite_win_rate"] - tab.iloc[0]["favorite_win_rate"]),
        "is_monotonic_expected_direction": is_monotonic,
        "monotonicity_fraction": round(mono_score, 3) if not np.isnan(mono_score) else np.nan,
    }
    return tab, summary


# For instability, lower values are generally better; reverse expected direction.
expected_ascending = {
    "risk_adjusted_edge": True,
    "quality_score": True,
    "coherence_score": True,
    "instability": False,
}

summaries = []
for m in metrics:
    if m not in cal_df.columns:
        continue
    tab, s = quantile_calibration_table(cal_df, m, q=10, ascending_good=expected_ascending[m])
    if tab is None:
        continue
    print(f"\n{m}: quantile calibration")
    display(tab)
    summaries.append(s)

if summaries:
    print("\nMonotonicity summary")
    display(pd.DataFrame(summaries).sort_values("metric").reset_index(drop=True))


risk_adjusted_edge: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(-2.409, -0.64]",259,161,0.622,98
1,"(-0.64, -0.49]",258,135,0.523,123
2,"(-0.49, -0.395]",258,154,0.597,104
3,"(-0.395, -0.321]",258,147,0.570,111
4,"(-0.321, -0.256]",258,144,0.558,114
5,"(-0.256, -0.198]",258,152,0.589,106
6,"(-0.198, -0.137]",258,134,0.519,124
7,"(-0.137, -0.0692]",258,134,0.519,124
8,"(-0.0692, 0.027]",258,142,0.550,116
9,"(0.027, 0.626]",258,151,0.585,107



quality_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(-0.00099223, 0.021]",259,154,0.595,105
1,"(0.021, 0.0338]",258,131,0.508,127
2,"(0.0338, 0.0428]",258,145,0.562,113
3,"(0.0428, 0.0507]",258,139,0.539,119
4,"(0.0507, 0.0567]",258,131,0.508,127
5,"(0.0567, 0.0646]",259,138,0.533,121
6,"(0.0646, 0.0767]",257,142,0.553,115
7,"(0.0767, 0.131]",258,163,0.632,95
8,"(0.131, 0.2]",258,148,0.574,110
9,"(0.2, 0.968]",258,163,0.632,95



coherence_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.256, 0.354]",259,143,0.552,116
1,"(0.354, 0.368]",258,144,0.558,114
2,"(0.368, 0.381]",258,135,0.523,123
3,"(0.381, 0.393]",258,146,0.566,112
4,"(0.393, 0.408]",258,142,0.550,116
5,"(0.408, 0.431]",258,136,0.527,122
6,"(0.431, 0.553]",258,136,0.527,122
7,"(0.553, 0.596]",258,167,0.647,91
8,"(0.596, 0.633]",258,156,0.605,102
9,"(0.633, 0.717]",258,149,0.578,109



instability: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.0375, 0.428]",259,151,0.583,108
1,"(0.428, 0.629]",258,129,0.500,129
2,"(0.629, 0.815]",258,141,0.547,117
3,"(0.815, 0.987]",258,148,0.574,110
4,"(0.987, 1.136]",258,147,0.570,111
5,"(1.136, 1.316]",258,153,0.593,105
6,"(1.316, 1.549]",258,140,0.543,118
7,"(1.549, 1.821]",258,148,0.574,110
8,"(1.821, 2.281]",258,143,0.554,115
9,"(2.281, 9.438]",258,154,0.597,104



Monotonicity summary


,metric,quantile_bins,rows_used,win_rate_first_bin,win_rate_last_bin,lift_last_minus_first,is_monotonic_expected_direction,monotonicity_fraction
0,coherence_score,10,2581,0.552,0.578,0.026,False,0.444
1,instability,10,2581,0.583,0.597,0.014,False,0.444
2,quality_score,10,2581,0.595,0.632,0.037,False,0.556
3,risk_adjusted_edge,10,2581,0.622,0.585,-0.037,False,0.556
